# Our Discogs Collections → Knowledge Graph

Fetch two vinyl collections from the Discogs API and merge them into one queryable knowledge graph with maplib.

In [11]:
import polars as pl
import urllib.request, urllib.parse, json, time
from maplib import Model

DISCOGS_TOKEN = "USYrpOyRKyHymBwPlTSjJMZFeftKDQbwxodogQHR"  # discogs.com/settings/developers
DISCOGS_USERS = ["veronika.heim", "pokkersidyll"]

base_url = "https://api.discogs.com"
headers = {
    "Authorization": f"Discogs token={DISCOGS_TOKEN}",
    "User-Agent": "maplib-collection/1.0",
}

## Fetch collections from Discogs

The collection API is paginated. We'll walk through all pages for each user and tag every release with its collector.

In [12]:
# Fetch all releases from a user's collection (all folders)
def fetch_collection(user, headers):
    items = []
    page = 1
    while True:
        url = f"{base_url}/users/{user}/collection/folders/0/releases?page={page}&per_page=100"
        req = urllib.request.Request(url, headers=headers)
        data = json.loads(urllib.request.urlopen(req).read())

        for item in data["releases"]:
            info = item["basic_information"]
            items.append({
                "release_id": info["id"],
                "title": info["title"],
                "artist": ", ".join(a["name"] for a in info["artists"]),
                "year": info["year"],
                "label": info["labels"][0]["name"] if info.get("labels") else None,
                "format": info["formats"][0]["name"] if info.get("formats") else None,
                "genres": ", ".join(info.get("genres", [])),
                "styles": ", ".join(info.get("styles", [])),
                "added": item.get("date_added", ""),
                "rating": item.get("rating", 0),
                "collector": user,
            })

        if page >= data["pagination"]["pages"]:
            break
        page += 1
        time.sleep(1)

    return pl.DataFrame(items)

# Fetch both collections and combine
all_collections = []
for user in DISCOGS_USERS:
    df = fetch_collection(user, headers)
    print(f"  {user}: {len(df)} releases")
    all_collections.append(df)

collection = pl.concat(all_collections)
print(f"\nTotal: {len(collection)} releases from {len(DISCOGS_USERS)} collectors")
collection.head(10)

  veronika.heim: 75 releases
  pokkersidyll: 462 releases

Total: 537 releases from 2 collectors


release_id,title,artist,year,label,format,genres,styles,added,rating,collector
i64,str,str,i64,str,str,str,str,str,i64,str
11849810,"""Film Festival Cannes""","""Jo Basile, Accordion And Orche…",0,"""Audio Fidelity""","""Vinyl""","""Stage & Screen""","""Soundtrack""","""2025-03-11T12:14:55-07:00""",0,"""veronika.heim"""
2786556,"""Helloween""","""Helloween""",1985,"""Banzai Records""","""Vinyl""","""Rock""","""Speed Metal, Heavy Metal""","""2023-12-05T08:07:55-08:00""",0,"""veronika.heim"""
3262647,"""Walls Of Jericho""","""Helloween""",1985,"""Banzai Records""","""Vinyl""","""Rock""","""Heavy Metal""","""2023-12-05T08:08:12-08:00""",0,"""veronika.heim"""
6275051,"""Lament""","""Einstürzende Neubauten""",2014,"""BMG""","""Vinyl""","""Electronic, Rock""","""Avantgarde, Experimental""","""2023-12-05T07:44:52-08:00""",0,"""veronika.heim"""
7657082,"""Keeper Of The Seven Keys (Part…","""Helloween""",2015,"""BMG""","""Vinyl""","""Rock""","""Heavy Metal, Power Metal""","""2023-12-05T08:06:53-08:00""",0,"""veronika.heim"""
391003,"""Demons And Wizards""","""Uriah Heep""",1972,"""Bronze""","""Vinyl""","""Rock""","""Prog Rock, Classic Rock""","""2025-04-26T13:16:40-07:00""",0,"""veronika.heim"""
5186505,"""Smule Smale Smile-viser (13 Fe…","""Bjørn Rønningen, Erling Bonde""",1978,"""CBS""","""Vinyl""","""Children's""","""""","""2023-12-05T08:02:19-08:00""",0,"""veronika.heim"""
14818790,"""III""","""Demons & Wizards""",2020,"""Century Media""","""Vinyl""","""Rock""","""Heavy Metal""","""2023-12-05T07:48:56-08:00""",0,"""veronika.heim"""
431665,"""Wednesday Morning, 3 A.M.""","""Simon & Garfunkel""",1965,"""Columbia""","""Vinyl""","""Rock, Folk, World, & Country""","""Folk, Folk Rock""","""2023-12-05T08:01:26-08:00""",0,"""veronika.heim"""


## Map to a knowledge graph

Artists and releases become entities. Each release links to its collector so we can query who owns what — and find shared records.

In [ ]:
m = Model()
ns = "http://example.org/music/"

# Load the ontology
m.read("data/rdf/ontology.ttl")

# Build IRIs for artists, releases, and collectors
mapped = collection.with_columns(
    (pl.lit(ns + "release/") + pl.col("release_id").cast(pl.Utf8)).alias("release_iri"),
    (pl.lit(ns + "band/") + pl.col("artist").str.replace_all(" ", "_")).alias("artist_iri"),
    (pl.lit(ns + "collector/") + pl.col("collector").str.replace_all(" ", "_")).alias("collector_iri"),
)

# Artist template
m.add_template("""
@prefix mu:<http://example.org/music/>.
@prefix xsd:<http://www.w3.org/2001/XMLSchema#>.

mu:Artist [
    ottr:IRI ?artist_iri,
    xsd:string ?name
] :: {
    ottr:Triple(?artist_iri, a,       mu:Band),
    ottr:Triple(?artist_iri, mu:name, ?name)
} .
""")

artists = mapped.select(
    pl.col("artist_iri"), pl.col("artist").alias("name")
).unique(subset=["artist_iri"])

m.map(ns + "Artist", artists)
print(f"Artists: {len(artists)} → {m.size()} triples")

In [ ]:
# Release template (now includes collector link)
m.add_template("""
@prefix mu:<http://example.org/music/>.
@prefix xsd:<http://www.w3.org/2001/XMLSchema#>.

mu:Release [
    ottr:IRI ?release_iri,
    xsd:string ?title,
    ottr:IRI ?artist_iri,
    xsd:long ?year,
    xsd:string ?label,
    xsd:string ?format,
    xsd:string ?genres,
    xsd:string ?styles,
    xsd:long ?release_id,
    xsd:long ?rating,
    ottr:IRI ?collector_iri,
    xsd:string ?collector
] :: {
    ottr:Triple(?release_iri, a,                  mu:Album),
    ottr:Triple(?release_iri, mu:title,           ?title),
    ottr:Triple(?release_iri, mu:artist,          ?artist_iri),
    ottr:Triple(?release_iri, mu:year,            ?year),
    ottr:Triple(?release_iri, mu:firstPressLabel, ?label),
    ottr:Triple(?release_iri, mu:firstPressFormat, ?format),
    ottr:Triple(?release_iri, mu:genre,           ?genres),
    ottr:Triple(?release_iri, mu:style,           ?styles),
    ottr:Triple(?release_iri, mu:discogsId,       ?release_id),
    ottr:Triple(?release_iri, mu:rating,          ?rating),
    ottr:Triple(?release_iri, mu:ownedBy,         ?collector_iri),
    ottr:Triple(?collector_iri, mu:name,          ?collector)
} .
""")

releases = mapped.select(
    "release_iri", "title", "artist_iri", "year", "label",
    "format", "genres", "styles", "release_id", "rating",
    "collector_iri", "collector",
)

m.map(ns + "Release", releases)
print(f"+ Releases: {m.size()} triples total")

## Query the collection

In [ ]:
# How many releases per artist (across both collections)?
m.query("""
    PREFIX mu: <http://example.org/music/>
    SELECT ?artist (COUNT(?r) AS ?releases)
    WHERE {
        ?r a mu:Album ; mu:artist ?b .
        ?b mu:name ?artist .
    }
    GROUP BY ?artist
    ORDER BY DESC(?releases)
""")

In [7]:
# Collection by decade
m.query("""
    PREFIX mu: <http://example.org/music/>
    SELECT ?decade (COUNT(?r) AS ?releases)
    WHERE {
        ?r a mu:Album ; mu:year ?y .
        FILTER(?y > 0)
        BIND((?y / 10) * 10 AS ?decade)
    }
    GROUP BY ?decade
    ORDER BY ?decade
""")

decade,releases
f64,u32
1960.0,3
1970.0,10
1980.0,9
1990.0,3
2000.0,10
2010.0,22
2020.0,13


In [8]:
# Collection by genre
m.query("""
    PREFIX mu: <http://example.org/music/>
    SELECT ?genre (COUNT(?r) AS ?releases)
    WHERE {
        ?r a mu:Album ; mu:genre ?genre .
    }
    GROUP BY ?genre
    ORDER BY DESC(?releases)
""")

genre,releases
str,u32
"""Rock""",28
"""Folk, World, & Country""",6
"""Pop, Folk, World, & Country""",4
"""Rock, Folk, World, & Country""",4
"""Pop""",3
…,…
"""Classical, Stage & Screen""",1
"""Funk / Soul""",1
"""Electronic, Jazz""",1


In [9]:
# Which labels do I buy from most?
m.query("""
    PREFIX mu: <http://example.org/music/>
    SELECT ?label (COUNT(?r) AS ?releases)
    WHERE {
        ?r a mu:Album ; mu:firstPressLabel ?label .
    }
    GROUP BY ?label
    ORDER BY DESC(?releases)
    LIMIT 15
""")

label,releases
str,u32
"""Strange Ways Records""",7
"""Pink Floyd Records""",6
"""Kirkelig Kulturverksted""",6
"""Mercury""",3
"""Nuclear Blast""",3
…,…
"""Talent (2)""",2
"""NO KIDDING""",2
"""BMG""",2


In [ ]:
# Artists you both collect — shared taste
m.query("""
    PREFIX mu: <http://example.org/music/>
    SELECT ?artist ?veronika_has ?pokkersidyll_has
    WHERE {
        {
            SELECT ?b (COUNT(DISTINCT ?r1) AS ?veronika_has)
            WHERE {
                ?r1 a mu:Album ; mu:artist ?b ; mu:ownedBy ?c1 .
                ?c1 mu:name "veronika.heim" .
            }
            GROUP BY ?b
        }
        {
            SELECT ?b (COUNT(DISTINCT ?r2) AS ?pokkersidyll_has)
            WHERE {
                ?r2 a mu:Album ; mu:artist ?b ; mu:ownedBy ?c2 .
                ?c2 mu:name "pokkersidyll" .
            }
            GROUP BY ?b
        }
        ?b mu:name ?artist .
    }
    ORDER BY DESC(?veronika_has)
""")

In [ ]:
# Releases per collector
m.query("""
    PREFIX mu: <http://example.org/music/>
    SELECT ?collector (COUNT(?r) AS ?releases)
    WHERE {
        ?r a mu:Album ; mu:ownedBy ?c .
        ?c mu:name ?collector .
    }
    GROUP BY ?collector
""")

In [ ]:
# For shared artists: albums only one of you owns — gift ideas!
m.query("""
    PREFIX mu: <http://example.org/music/>
    SELECT ?artist ?title ?owned_by
    WHERE {
        # Find artists in both collections
        ?r1 a mu:Album ; mu:artist ?b ; mu:ownedBy ?c1 .
        ?c1 mu:name "veronika.heim" .
        ?r2 a mu:Album ; mu:artist ?b ; mu:ownedBy ?c2 .
        ?c2 mu:name "pokkersidyll" .

        # Now find releases of that artist owned by only one collector
        ?r a mu:Album ;
           mu:title ?title ;
           mu:artist ?b ;
           mu:ownedBy ?owner .
        ?owner mu:name ?owned_by .
        FILTER NOT EXISTS {
            ?r mu:ownedBy ?other_owner .
            ?other_owner mu:name ?other_name .
            FILTER(?other_name != ?owned_by)
        }
        ?b mu:name ?artist .
    }
    ORDER BY ?artist ?title
""")

## Explore

Interactive graph browser over both collections merged. Try searching for a shared artist to see both collectors' releases.

In [10]:
# Add labels for the explorer
m.insert("""
    PREFIX mu:   <http://example.org/music/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    CONSTRUCT { ?b rdfs:label ?name }
    WHERE     { ?b mu:name ?name }
""")
m.insert("""
    PREFIX mu:   <http://example.org/music/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    CONSTRUCT { ?r rdfs:label ?title }
    WHERE     { ?r mu:title ?title }
""")

server = m.explore(port=8002)
print("Explorer at http://localhost:8002")

Access graph explorer on http://localhost:8002
Explorer at http://localhost:8002


In [ ]:
server.stop()